In [ ]:
%load_ext autoreload
%autoreload 2

# system imports
import sys
import os

# add the parent directory of 'notebooks' to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))  # move one level up
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [ ]:
import torch
import optuna
from pathlib import Path
from solver import Solver, TrialMetric
from griddy_tuna import hit_griddy, SearchMethod, TrialMetric
from models.griddy_model import GriddyModel
from data.data_loader import MirDataProcessor
from optuna.visualization import plot_optimization_history, plot_contour, plot_parallel_coordinate, plot_slice

In [ ]:
device = 'cuda'

In [ ]:
# if you have already ran the downloader, change the value of download to False
download = False

# reprocess for different dataset type while bypassing download
reprocess = True

# download and build useable train/test data out of the MIR Billboard dataset
data_processer = MirDataProcessor(output_dir=None, download=download, batch_size=64) # your notebook should be in its own directory to begin with, this should create the "data" folder inside that
if download:
    data_processer.process_billboard_data(log_fail_only=False) # you may need to reprocess the downloaded data into sequential or tabular based on your model
if reprocess:
    data_processer.dataset.download(partial_download=['metadata'])
    data_processer.process_billboard_data(combined_notation=True, chord_vocab='majmin7inv', log_fail_only=False)
    # combined notation is standard billboard notation (C:maj), setting False creates separate CSVs for root and chord_class

# dataset options: 'combined', 'root', 'chord_class'
train_loader, test_loader, num_classes = data_processer.build_data_loaders(device=device, dataset='combined', nrows=None) # set nrows to shrink dataset for testing

print(f"Number of classes: {num_classes}")

In [ ]:
data_processer.process_billboard_data()

In [ ]:
# NOTE: do not assume these values are anything but trash, they're just here for testing

SOLVER_PARAMS = {
    Solver : {
        "device": device,
        "batch_size": 64,
        "epochs": 10,
        "early_stop_epochs": 0, # early stop after n epochs without improvement, 0 to disable
        "warmup_epochs": 0, # 0 to disable
        "dtype": "float16",
        "train_dataloader": train_loader, # must be DataLoader object
        "valid_dataloader": test_loader, # must be DataLoader object
        "direction": "minimize" # must specify this, even if not used by solver
    }
}

MODEL_PARAMS = {
    CRNNModel: {
        "input_features": [24],
        "num_classes": [num_classes],
        "hidden_size": [128],
    }
}

OPTIM_PARAMS = {
    torch.optim.SGD : {
        "lr": [0.001, 0.1, SearchMethod.LOG_UNIFORM],
        "momentum": [0.9, 0.99, SearchMethod.UNIFORM],
        "weight_decay": [0.001, 0.000001, SearchMethod.LOG_UNIFORM],
    },
    torch.optim.Adam : {
        "lr": [0.001, 0.1, SearchMethod.LOG_UNIFORM],
    }
}

SCHED_PARAMS = {
    torch.optim.lr_scheduler.CosineAnnealingWarmRestarts : {
        "T_0": [10],
    },
    torch.optim.lr_scheduler.StepLR : {
        "step_size": [10],
        "gamma" : [0.1],
    }
}

CRITERION_PARAMS = {
    torch.nn.CrossEntropyLoss : {}
}

PARAM_SET = {
    "solver": SOLVER_PARAMS,
    "model" : MODEL_PARAMS,
    "optim" : OPTIM_PARAMS,
    "sched" : SCHED_PARAMS,
    "criterion" : CRITERION_PARAMS,
}

In [ ]:
study_name = "my_study"
output_folder = Path("griddy") # relative to working directory

In [ ]:
study = hit_griddy(study_name, param_set=PARAM_SET, out_dir=output_folder, trial_metric=TrialMetric.LOSS, n_trials=60, n_jobs=6, prune=False, resume=False)
# NOTE: modest values of n_trials and n_jobs set here for testing, set your values accordingly
# trial_metric can be LOSS or ACCURACY, can add others to solver and expand options

In [ ]:
### NORMAL RUN ###

model = list(MODEL_PARAMS.keys())[0](**MODEL_PARAMS[list(MODEL_PARAMS.keys())[0]])
optimizer = list(OPTIM_PARAMS.keys())[0](**(OPTIM_PARAMS[list(OPTIM_PARAMS.keys())[0]] | {'params': model.parameters()}))
scheduler = list(SCHED_PARAMS.keys())[0](**(SCHED_PARAMS[list(SCHED_PARAMS.keys())[0]] | {'optimizer': optimizer}))
criterion = list(CRITERION_PARAMS.keys())[0](**CRITERION_PARAMS[list(CRITERION_PARAMS.keys())[0]])
solver = Solver(**(SOLVER_PARAMS[Solver] | {'model': model, 'optimizer': optimizer, 'scheduler': scheduler, 'criterion': criterion}))

solver.train_and_evaluate(plot_results=True)

In [ ]:
full_path = os.path.join(output_folder, f"{study_name}.db")
storage_path = f'sqlite:///{full_path}'

saved_study = optuna.load_study(study_name, storage_path)

In [ ]:
df = saved_study.trials_dataframe(attrs=['number', 'value', 'params', 'state'])
print(df)

In [ ]:
# plotting the optimization history
plot_optimization_history(saved_study)
# displaying the contour plot of parameter relationships
plot_contour(saved_study)
# visualizing high-dimensional relationships
plot_parallel_coordinate(saved_study)
# slice plot
plot_slice(saved_study)